# **Installing packages related to their import**

In [ ]:
!pip install tqdm
!pip install retina-face

In [ ]:
from retinaface import RetinaFace
import cv2 as cv
from tqdm import tqdm
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path


Mounted at /content/drive


# **Face recognition from real and fake frames**

In [ ]:
#Face recognition function
def detect_face(path_face,min_confidence=0.8):

  image = cv.imread(str(path_face))
  if image is None:
    return None

  faces = RetinaFace.detect_faces(image)
  if not isinstance(faces,dict):
    return None

  best_face = max(

      faces.values(),
      key=lambda x: x['score']
  )
#If the model is confident it found more than 80% of faces, it will return them
  if best_face['score'] < min_confidence:
    return None
#Cropping the detected face
  x1,y1,x2,y2 =best_face["facial_area"]
  face=image[y1:y2,x1:x2]
  return face



In [ ]:
#Saving the detected image
def Save_face(frame_path,output_dir):
  n_saved, n_skipped = 0, 0
  for p in tqdm(frame_path):
    try:
      face = detect_face(p)
      if face is not None:
        face=cv.resize(face , (224,224))
        save_path = output_dir / p.name
        if save_path.exists():
          n_skipped +=1
          continue
        ok = cv.imwrite(str(save_path), face)
        if ok:
          n_saved += 1
        else:
          print(f"Failed to save: {p}")
          n_skipped += 1
      else:
        print(f"No face detected: {p}")
        n_skipped += 1
    except Exception as e:
      print(f"Error processing {p}: {e}")
      n_skipped += 1
    print(f"{output_dir} -> saved: {n_saved}, skipped: {n_skipped}")

In [ ]:
frame_paths_train_real = sorted(Path("/content/drive/MyDrive/frame_train/frame_real").rglob("*.jpg"))
frame_paths_train_fake = sorted(Path("/content/drive/MyDrive/frame_train/frame_fake").rglob("*.jpg"))
frame_path_test_real = sorted(Path("/content/drive/MyDrive/frame_test/farme_real").rglob("*.jpg"))
frame_path_test_fake = sorted(Path("/content/drive/MyDrive/frame_test/frame_fake").rglob("*.jpg"))

output_dir_train_real = Path("/content/drive/MyDrive/face_detection_train/face_real")
output_dir_train_fake = Path("/content/drive/MyDrive/face_detection_train/face_fake")
output_dir_test_real = Path("/content/drive/MyDrive/face_detection_test/face_real")
output_dir_test_fake = Path("/content/drive/MyDrive/face_detection_test/face_fake")
for d in [output_dir_train_real,output_dir_train_fake,output_dir_test_real,output_dir_test_fake ]:
    d.mkdir(parents=True, exist_ok=True)

Save_face(frame_paths_train_real, output_dir_train_real)
Save_face(frame_paths_train_fake, output_dir_train_fake)
Save_face(frame_path_test_real,output_dir_test_real)
Save_face(frame_path_test_fake,output_dir_test_fake)


  0%|          | 0/3893 [00:00<?, ?it/s]

26-06-22 05:50:33 - Directory /root/.deepface created
26-06-22 05:50:33 - Directory /root/.deepface/weights created
26-06-22 05:50:33 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /root/.deepface/weights/retinaface.h5

  0%|          | 0.00/119M [00:00<?, ?B/s]
 14%|█▍        | 16.8M/119M [00:00<00:00, 167MB/s]
 42%|████▏     | 50.3M/119M [00:00<00:00, 266MB/s]
 72%|███████▏  | 84.9M/119M [00:00<00:00, 302MB/s]
100%|██████████| 119M/119M [00:00<00:00, 275MB/s]
  1%|          | 30/3893 [05:33<11:54:51, 11.10s/it]


KeyboardInterrupt: 